# Eksperimen Machine Learning - Dataset Sports Car
**Nama Siswa:** Nama-siswa  
**Dataset:** Sports Car Dataset (dataset_raw.csv)  
**Tujuan:** Memprediksi harga mobil (Price in USD) berdasarkan fitur teknikal

---
## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

print('Library berhasil diimport!')

---
## 2. Data Loading

In [ ]:
# Load dataset raw
df = pd.read_csv('../dataset_raw.csv')
print(f'Dataset berhasil dimuat!')
print(f'Shape: {df.shape}')
print(f'Jumlah baris: {df.shape[0]}, Jumlah kolom: {df.shape[1]}')

In [ ]:
# Tampilkan 5 baris pertama
df.head()

In [ ]:
# Tampilkan info kolom
df.info()

---
## 3. Exploratory Data Analysis (EDA)

### 3.1 Statistik Deskriptif

In [ ]:
# Konversi kolom numerik terlebih dahulu untuk keperluan describe
df_temp = df.copy()
numeric_cols = ['Engine Size (L)', 'Horsepower', 'Torque (lb-ft)', '0-60 MPH Time (seconds)']
for col in numeric_cols:
    df_temp[col] = pd.to_numeric(df_temp[col], errors='coerce')
df_temp['Price (in USD)'] = df_temp['Price (in USD)'].str.replace(',', '').astype(float)

df_temp.describe()

### 3.2 Cek Missing Values

In [ ]:
# Cek missing values sebelum konversi
print('Missing values (raw):')
print(df.isnull().sum())
print()

# Cek missing values setelah konversi
print('Missing values (setelah konversi numerik):')
print(df_temp.isnull().sum())

# Visualisasi missing values
plt.figure(figsize=(10, 4))
missing_pct = df_temp.isnull().sum() / len(df_temp) * 100
missing_pct[missing_pct > 0].plot(kind='bar', color='salmon')
plt.title('Persentase Missing Values per Kolom')
plt.ylabel('Persentase (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 3.3 Cek Duplikasi

In [ ]:
print(f'Jumlah data duplikat: {df.duplicated().sum()}')
print(f'Persentase duplikat: {df.duplicated().sum()/len(df)*100:.2f}%')

### 3.4 Distribusi Target Variable (Price)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribusi price sebelum log
axes[0].hist(df_temp['Price (in USD)'].dropna(), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribusi Harga Mobil (Original)')
axes[0].set_xlabel('Harga (USD)')
axes[0].set_ylabel('Frekuensi')

# Distribusi price setelah log
axes[1].hist(np.log1p(df_temp['Price (in USD)'].dropna()), bins=50, color='teal', edgecolor='white')
axes[1].set_title('Distribusi Harga Mobil (Log Transform)')
axes[1].set_xlabel('Log(Harga + 1)')
axes[1].set_ylabel('Frekuensi')

plt.tight_layout()
plt.show()

### 3.5 Distribusi Fitur Numerik

In [ ]:
num_features = ['Engine Size (L)', 'Horsepower', 'Torque (lb-ft)', '0-60 MPH Time (seconds)', 'Year']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].hist(df_temp[col].dropna(), bins=30, color='cornflowerblue', edgecolor='white')
    axes[i].set_title(f'Distribusi {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frekuensi')

axes[-1].axis('off')
plt.tight_layout()
plt.show()

### 3.6 Top 10 Car Makes

In [ ]:
plt.figure(figsize=(12, 5))
top_makes = df['Car Make'].value_counts().head(10)
top_makes.plot(kind='bar', color='mediumseagreen', edgecolor='white')
plt.title('Top 10 Car Makes Terbanyak dalam Dataset')
plt.xlabel('Car Make')
plt.ylabel('Jumlah')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 3.7 Korelasi Antar Fitur Numerik

In [ ]:
plt.figure(figsize=(10, 7))
corr_matrix = df_temp[num_features + ['Price (in USD)']].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, square=True)
plt.title('Heatmap Korelasi Fitur Numerik')
plt.tight_layout()
plt.show()

### 3.8 Scatter Plot: Horsepower vs Price

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df_temp['Horsepower'], df_temp['Price (in USD)'], alpha=0.4, color='coral')
plt.title('Horsepower vs Price')
plt.xlabel('Horsepower')
plt.ylabel('Price (USD)')
plt.tight_layout()
plt.show()

### 3.9 Cek Outlier dengan Boxplot

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].boxplot(df_temp[col].dropna())
    axes[i].set_title(f'Boxplot {col}')
    axes[i].set_ylabel(col)

axes[-1].axis('off')
plt.tight_layout()
plt.show()

---
## 4. Data Preprocessing

### 4.1 Konversi Tipe Data

In [ ]:
df_proc = df.copy()

# Konversi kolom numerik
numeric_cols = ['Engine Size (L)', 'Horsepower', 'Torque (lb-ft)', '0-60 MPH Time (seconds)']
for col in numeric_cols:
    df_proc[col] = pd.to_numeric(df_proc[col], errors='coerce')

# Konversi Price: hilangkan koma lalu jadikan float
df_proc['Price (in USD)'] = df_proc['Price (in USD)'].str.replace(',', '').astype(float)

print('Konversi tipe data selesai.')
print(df_proc.dtypes)

### 4.2 Hapus Duplikat

In [ ]:
sebelum = df_proc.shape[0]
df_proc = df_proc.drop_duplicates()
sesudah = df_proc.shape[0]
print(f'Baris sebelum drop duplikat : {sebelum}')
print(f'Baris setelah drop duplikat  : {sesudah}')
print(f'Jumlah duplikat yang dihapus : {sebelum - sesudah}')

### 4.3 Penanganan Missing Values

In [ ]:
print('Missing values sebelum penanganan:')
print(df_proc.isnull().sum())

# Isi missing value numerik dengan median
for col in numeric_cols:
    median_val = df_proc[col].median()
    df_proc[col] = df_proc[col].fillna(median_val)
    print(f'  [{col}] missing diisi dengan median: {median_val:.2f}')

print()
print('Missing values setelah penanganan:')
print(df_proc.isnull().sum())

### 4.4 Penanganan Outlier (IQR Method)

In [ ]:
def remove_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = len(data)
    data = data[(data[column] >= lower) & (data[column] <= upper)]
    after = len(data)
    print(f'  [{column}] dihapus {before-after} outlier (lower={lower:.2f}, upper={upper:.2f})')
    return data

print('Penanganan outlier dengan IQR:')
print(f'Baris sebelum: {len(df_proc)}')

outlier_cols = ['Horsepower', 'Torque (lb-ft)', 'Engine Size (L)', 'Price (in USD)']
for col in outlier_cols:
    df_proc = remove_outliers_iqr(df_proc, col)

print(f'Baris sesudah: {len(df_proc)}')

### 4.5 Encoding Fitur Kategorikal

In [ ]:
le_make = LabelEncoder()
le_model = LabelEncoder()

df_proc['Car Make Encoded'] = le_make.fit_transform(df_proc['Car Make'])
df_proc['Car Model Encoded'] = le_model.fit_transform(df_proc['Car Model'])

# Drop kolom kategorikal asli
df_proc = df_proc.drop(columns=['Car Make', 'Car Model'])

print('Encoding selesai.')
print(df_proc.head())

### 4.6 Feature Scaling (StandardScaler)

In [ ]:
# Pisahkan fitur dan target
target = 'Price (in USD)'
feature_cols = [col for col in df_proc.columns if col != target]

X = df_proc[feature_cols]
y = df_proc[target]

# Scaling fitur
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

print('Scaling selesai.')
print('Shape fitur:', X_scaled.shape)
print('Shape target:', y.shape)
X_scaled.describe().round(3)

### 4.7 Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f'X_train shape : {X_train.shape}')
print(f'X_test shape  : {X_test.shape}')
print(f'y_train shape : {y_train.shape}')
print(f'y_test shape  : {y_test.shape}')

### 4.8 Simpan Dataset Preprocessing

In [ ]:
import os
os.makedirs('dataset_preprocessing', exist_ok=True)

# Gabungkan kembali X_scaled dan y untuk disimpan
df_final = X_scaled.copy()
df_final[target] = y.reset_index(drop=True)

df_final.to_csv('dataset_preprocessing/dataset_preprocessing.csv', index=False)
print(f'Dataset preprocessing tersimpan!')
print(f'Shape final: {df_final.shape}')
df_final.head()

---
## 5. Kesimpulan

| Tahap | Keterangan |
|-------|------------|
| Data Raw | 1007 baris, 8 kolom |
| Setelah Drop Duplikat | 715 baris |
| Setelah Handle Missing | Median imputation pada 4 kolom numerik |
| Setelah Remove Outlier | IQR method pada 4 kolom |
| Encoding | LabelEncoder pada Car Make & Car Model |
| Scaling | StandardScaler pada seluruh fitur |
| Split | 80% train, 20% test |

Dataset siap digunakan untuk pelatihan model machine learning.